# TimesFM 3 P0b: finite-candidate oracle screen

Runs the matched P0b candidate-oracle screen through the pinned official FEV TimesFM-3 wrapper. TimesFM 3 weights are restricted to academic, non-commercial use. Run this only after the Chronos-2 P0b report passes review.

Use an **A100** if available. Completed `(task, origin)` units are written atomically to Google Drive and safely resume after disconnects.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
FEV_CHECKOUT = Path('/content/fev-pinned')
FEV_COMMIT = '38007871dcf6dc6b04aed3a54d9cd86678d48d0b'
TIMESFM_COMMIT = '20191171b74f51bfead932b6b8d0c8f515e70f63'
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if not FEV_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', '--filter=blob:none',
         'https://github.com/autogluon/fev.git', str(FEV_CHECKOUT)],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(FEV_CHECKOUT), 'checkout', FEV_COMMIT],
        check=True,
    )
else:
    observed = subprocess.check_output(
        ['git', '-C', str(FEV_CHECKOUT), 'rev-parse', 'HEAD'], text=True
    ).strip()
    assert observed == FEV_COMMIT, observed
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        f'git+https://github.com/autogluon/fev.git@{FEV_COMMIT}',
        ('timesfm[torch] @ git+https://github.com/google-research/'
         f'timesfm.git@{TIMESFM_COMMIT}'),
    ],
    check=True,
)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
covsafe = importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Pinned FEV checkout:', FEV_CHECKOUT)
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())
print('HF token available:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'Select Runtime > Change runtime type > GPU, then restart.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path(
    '/content/drive/MyDrive/covariate-safe-tsfm/private_manifests'
)
P0A_ROOT = PRIVATE_ROOT / 'p0a'
P0B_ROOT = PRIVATE_ROOT / 'p0b'
assert P0A_ROOT.exists(), 'P0a artifacts are missing from Google Drive.'
P0B_ROOT.mkdir(parents=True, exist_ok=True)
print('P0a input root:', P0A_ROOT)
print('P0b durable root:', P0B_ROOT)

In [ ]:
import json

from covsafe.p0b import EXPECTED_P0B_CONFIG_HASH
from covsafe.timesfm3_p0b import run_timesfm3_p0b

print('Frozen P0b config hash:', EXPECTED_P0B_CONFIG_HASH)
report = run_timesfm3_p0b(
    REPO, FEV_CHECKOUT, P0A_ROOT, P0B_ROOT
)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

## Return artifact

Send the final JSON containing `h2_screen`. `HF_TOKEN` is strongly recommended because this stage is longer; a read-only token is sufficient. If disconnected, rerun all cells and completed origins print `RESUME`.